In [0]:
from pyspark.sql import functions as F
from config.f1_config import CATALOG, SILVER_SCHEMA, GOLD_SCHEMA
GOLD_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.drivers"

In [0]:
df_silver_drivers = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.drivers")
df_silver_results = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.results")


In [0]:
def build_dim_drivers(df_silver_drivers):
    active_driver_ids = df_silver_results.select("Driver_Id").distinct()
    return(
        df_silver_drivers
        .join(active_driver_ids, on="Driver_Id", how="inner")
        .withColumn("Full_Name", F.concat_ws(" ", F.col("First_Name"), F.col("Last_Name")))
        .select(
            "Driver_Id", "Driver_Code", "First_Name", "Last_Name", "Full_Name", "Date_Of_Birth", "Nationality", "Driver_Number"
            )
)
df_gold_dim_drivers = build_dim_drivers(df_silver_drivers)

In [0]:
df_gold_dim_drivers.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(GOLD_TABLE)